In [1]:
import math
import torch
import torch.nn.functional as F

In [3]:
def attention(q, k, v):
    scores = q @ k.T # scoring
    scaled = scores / math.sqrt(k.shape[1]) # scaling
    mask = torch.tril(torch.ones(len(q), len(q)))
    masked = scaled.masked_fill(mask == 0, float('-inf'))
    weights = F.softmax(masked, dim=-1)
    return weights @ v

### Three ways to turn words into numbers

1. Word IDs - one number per word.\
Problem: the number is arbitrary and means nothing.

2. The embedding - a row of numbers per word, close rows for close meanings.\
Problem: Words are independent of context. The same word gets the same row in EVERY sentence.

3. The Attention. Rewrite each word from its neighbors. Accounting the CONTEXT.\
Problem: it can peek word after it, so CONTEXT TIMELINE IS WRONG, but it can be easily solved using MASKING.

In [4]:
# Word IDs
vocab = ["the", "crane", "lifted", "steel", "ate", "fish"]
word_to_id = {w: i for i, w in enumerate(vocab)}
word_to_id

{'the': 0, 'crane': 1, 'lifted': 2, 'steel': 3, 'ate': 4, 'fish': 5}

In [9]:
for s in ["the crane lifted steel", "the crane ate fish"]:
    print(s, "->", [word_to_id[w] for w in s.split()])

the crane lifted steel -> [0, 1, 2, 3]
the crane ate fish -> [0, 1, 4, 5]


In [10]:
vocab = ["the", "crane", "lifted", "steel", "ate", "fish"]
vocab2 = ["steel", "the", "lifted", "crane", "ate", "fish"]
word_to_id = {w: i for i, w in enumerate(vocab)}
word_to_id2 = {w: i for i, w in enumerate(vocab2)}
word_to_id, word_to_id2

({'the': 0, 'crane': 1, 'lifted': 2, 'steel': 3, 'ate': 4, 'fish': 5},
 {'steel': 0, 'the': 1, 'lifted': 2, 'crane': 3, 'ate': 4, 'fish': 5})

In [18]:
[word_to_id[w] for w in "the crane lifted steel".split()],\
[word_to_id2[w] for w in "the crane lifted steel".split()]

([0, 1, 2, 3], [1, 3, 2, 0])

In [19]:
# Embeddings - which lears using model for it, but this is example

In [23]:
# a tiny embedding table by hand
emb = torch.tensor([[0.1, 0.1],  # the
                    [0.7, 0.7],  # crane
                    [0.1, 0.9],  # lifted
                    [0.0, 0.9],  # steel
                    [0.9, 0.1],  # at
                    [0.9, 0.0]]) # fish

In [22]:
emb[word_to_id["crane"]]

tensor([0.7000, 0.7000])

In [27]:
# But thats a problem: in embeddings there is ONLY ONE row for word "crane".
# Excluding CONTEXT.

Scaled Dot-Product Attention (SDPA)
$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}+\text{M}\right)V $$

1. Score - match each question against every reply
2. Scale - divide every score by sqrt(d)
3. Mask - hide the words that come later
4. Normalize - turn scores into shares that add to one
5. Mix It - blend the offers by those shares

In [30]:
q = torch.tensor([[0.2, 0.2], [1.4, 1.4], [0.2, 1.8], [0.0, 1.8]]) # q, the queries
k = torch.tensor([[0.05, 0.05], [0.20, 0.20], [0.05, 0.45], [0.00, 0.45]]) # k, the keys
v = torch.tensor([[0.1, 0.1], [0.7, 0.7],[0.1, 0.9], [0.0, 0.9]]) # v, the value
q, k, v

(tensor([[0.2000, 0.2000],
         [1.4000, 1.4000],
         [0.2000, 1.8000],
         [0.0000, 1.8000]]),
 tensor([[0.0500, 0.0500],
         [0.2000, 0.2000],
         [0.0500, 0.4500],
         [0.0000, 0.4500]]),
 tensor([[0.1000, 0.1000],
         [0.7000, 0.7000],
         [0.1000, 0.9000],
         [0.0000, 0.9000]]))

In [31]:
# crane asks loud -> q(1.4, 1.4)

In [34]:
scores = q @ k.T
scores

tensor([[0.0200, 0.0800, 0.1000, 0.0900],
        [0.1400, 0.5600, 0.7000, 0.6300],
        [0.1000, 0.4000, 0.8200, 0.8100],
        [0.0900, 0.3600, 0.8100, 0.8100]])